# 19 · Can the market factors be traded after costs, without RL? (E19)

E18 showed that the E16 agents did not use the 8 market factors to time the market: they took long-lasting
directional bets. Before spending more GPU time on RL, this notebook asks whether the factors carry a signal that
**any** simple rule can turn into profit after costs. If not, no agent trained on them can either.

The factors are the E15 GRU's forecasts of the 8 trend labels: factor *j* predicts how the smoothed price
10, 30, 60, 120, 200, 300, 500 or 900 one-minute bars ahead compares with the price now (values in [−1, 1]).

1. **Edge.** BTC's forward return over each factor's horizon, by how strong the factor is. A rule can only make
   money if the strongest signals precede moves larger than the **round-trip cost of 0.24 %** (2 × (0.10 % fee +
   0.02 % slippage)).
2. **Threshold rules**, traded in the same environment as the agents (same costs, orders at the next bar, a
   decision every 5 minutes): long when the signal is above *k*, short when below −*k*, otherwise keep the
   position ("hold") or go flat ("flat"), optionally keeping each new position for at least the factor's horizon.
   *k* is a train quantile of |signal|. 9 signals (8 factors + their average) × 7 thresholds × 2 modes × 2 holding
   rules = 252 rules, each scored with and without costs.

**Splits.** Rules are chosen on **train** and judged on **validation**. Test and stress_bear are not loaded.
Two caveats: the GRU was fitted on train, so its train forecasts are in-sample (better than they will be on
new data); and validation already chose the GRU's settings (E15) and the agents' checkpoints (E16), so it is not
a fresh hold-out either. Both make the factors look better than they are, never worse.

CPU runtime is enough (about 5-10 minutes). Writes `thesis_rl/initiation/e19_factor_rules/` and `report/e19_*`.

In [ ]:
# ---- setup: Drive, code, packages (identical in every notebook) -------------------------
BRANCH = "exp/14-initiation"
REPO = "https://github.com/taimoor-ahmed-1/FinAI-CryptoRaiders.git"
DRIVE_ROOT = "/content/drive/MyDrive/thesis_rl"      # data in, every output out - survives disconnects

import os, sys, subprocess
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = "/content/FinAI-CryptoRaiders"
    if not os.path.exists(REPO_DIR):
        # sparse clone: only experiments/ is checked out - the repo's data folders are never downloaded
        subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout", "--branch", BRANCH, REPO, REPO_DIR], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "sparse-checkout", "set", "--no-cone", "/experiments/"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "-B", BRANCH, "FETCH_HEAD"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/experiments/requirements-colab.txt"], check=True)
    EXP = f"{REPO_DIR}/experiments"
else:                                                # local run from a checkout: notebooks/ -> experiments/
    EXP = os.path.abspath("..")
    DRIVE_ROOT = os.environ.get("THESIS_RL_ROOT", os.path.abspath("../../thesis_rl_outputs"))
sys.path.insert(0, EXP)
os.chdir(EXP)

import torch
from rltrade import paths, seeding
L = paths.layout(DRIVE_ROOT)
print("code:", BRANCH, seeding.git_commit(EXP), "| device:", "cuda - " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

In [ ]:
seeding.record_environment(L.report / "environment", "19_initiation_factor_rules")

In [ ]:
import json, time, zipfile, numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from rltrade import factor_rule as FR, results as R
from rltrade.env import EnvConfig, Market, run_policy
from rltrade.baselines import buy_and_hold_policy
from rltrade.metrics import compute_metrics

INIT = L.root / "initiation"
REP, OUT = INIT / "report", INIT / "e19_factor_rules"
for p in (REP, OUT): p.mkdir(parents=True, exist_ok=True)
LOCAL = Path("/content/initiation_prepared") if "google.colab" in sys.modules else INIT / "prepared"
ZIP = INIT / "initiation_prepared.zip"
PREP = LOCAL / "fixed_plus"
SPLITS = ("train", "val")                           # test and stress_bear are not loaded
NEEDED = [f"fixed_plus/market_{s}.npz" for s in SPLITS] + ["fixed_plus/meta.json"]
if not all((LOCAL / n).exists() for n in NEEDED):
    assert ZIP.exists(), f"upload initiation_prepared.zip to {ZIP}"
    with zipfile.ZipFile(ZIP) as z: z.extractall(LOCAL, members=NEEDED)
    print("unzipped", NEEDED, "->", LOCAL)

sel = json.loads((INIT / "selected_agent_e16.json").read_text())
def on_drive(p):                                    # paths written on another machine: same place under initiation/
    p = Path(p)
    return p if p.exists() else INIT.joinpath(*p.parts[p.parts.index("initiation") + 1:])
F = on_drive(sel["factor_dir"])
NET = EnvConfig(state="A")
GROSS = EnvConfig(state="A", fee_rate=0.0, slippage=0.0)
MK = {s: Market(PREP, F, s, NET, "cpu") for s in SPLITS}
PRICE = {s: np.load(PREP / f"market_{s}.npz")["midpoint"].astype(np.float64) for s in SPLITS}
SIG = {s: FR.signals(np.load(F / f"factors_{s}.npy")) for s in SPLITS}
for s in SPLITS:
    assert len(SIG[s]) == len(PRICE[s]), (s, SIG[s].shape, PRICE[s].shape)
BH = {s: compute_metrics(run_policy(buy_and_hold_policy, MK[s], NET), NET.periods_per_year) for s in SPLITS}
print(f"factors {F} | " + " | ".join(f"{s}: {len(PRICE[s]):,} bars, buy and hold Sharpe {BH[s]['sharpe']:+.2f}" for s in SPLITS))

## 1 · Edge: BTC's move over each factor's horizon, by signal strength

In [ ]:
QS = (0.01, 0.05, 0.10, 0.30, 0.50, 0.70, 0.90, 0.95, 0.99)
BINS = ["<1%", "1-5%", "5-10%", "10-30%", "30-50%", "50-70%", "70-90%", "90-95%", "95-99%", ">99%"]
edge_rows, ic_rows = [], []
for j, name in enumerate(FR.SIGNALS):
    h = FR.SIGNAL_HORIZON[j]
    cuts = np.quantile(SIG["train"][:, j], QS)                       # cut points from train
    for s in SPLITS:
        fwd = FR.forward_returns(PRICE[s], h)
        mean, cnt = FR.edge(SIG[s][:, j], fwd, cuts)
        edge_rows.append({"split": s, "signal": name, "horizon_bars": h, **dict(zip(BINS, mean))})
        rho, p, n = FR.ic(SIG[s][:, j], fwd, h)
        ic_rows.append({"split": s, "signal": name, "horizon_bars": h, "ic": rho, "ic_p": p, "n": n,
                        "top 1% move %": mean[-1], "bottom 1% move %": mean[0],
                        "half spread 1% %": (mean[-1] - mean[0]) / 2, "half spread 10% %": (mean[-3:].mean() - mean[:3].mean()) / 2})
edge_df, ic_df = pd.DataFrame(edge_rows), pd.DataFrame(ic_rows)
edge_df.to_csv(OUT / "edge_by_bin.csv", index=False); ic_df.to_csv(OUT / "ic.csv", index=False)
print(f"mean forward return (%) over each signal's horizon, by train percentile of the signal; round-trip cost {100 * FR.ROUND_TRIP_COST:.2f} %")
for s in SPLITS:
    print(f"\n--- {s}\n" + edge_df[edge_df.split == s].drop(columns="split").round(3).to_string(index=False))
print("\nIC = Spearman correlation with the forward return, one sample per horizon (non-overlapping)")
print("half spread = (move after the strongest long signals - move after the strongest short signals) / 2:")
print("the gross gain of one round trip on those signals; it has to beat 0.24 %")
print(ic_df.round(4).to_string(index=False))

## 2 · Threshold rules in the trading environment (252 rules, with and without costs)

In [ ]:
GRID = FR.make_grid(SIG["train"])
res = pd.DataFrame(GRID)
t0 = time.time()
for s in SPLITS:
    for tag, cfg in (("net", NET), ("gross", GROSS)):
        m = FR.run_rules(GRID, MK[s], cfg)
        for k in ("sharpe", "cum_return_pct", "max_drawdown_pct", "num_trades", "costs_pct", "exposure_pct", "long_pct", "ruined"):
            res[f"{s}_{tag}_{k}"] = m[k]
        print(f"  {s} {tag}: {len(GRID)} rules in {time.time() - t0:.0f} s")
res.to_csv(OUT / "rules.csv", index=False)

In [ ]:
show = ["signal", "q", "mode", "min_hold", "train_net_sharpe", "val_net_sharpe", "val_net_cum_return_pct",
        "val_net_max_drawdown_pct", "val_net_num_trades", "val_net_costs_pct", "val_net_long_pct", "val_gross_sharpe"]
elig = res[res["train_net_num_trades"] >= 5]           # same eligibility as the RL checkpoints: a rule must trade
best_per_signal = res.loc[elig.groupby("signal", sort=False)["train_net_sharpe"].idxmax(), show]
best = res.loc[elig["train_net_sharpe"].idxmax()]
bh_val = BH["val"]["sharpe"]
dist = pd.DataFrame([{
    "costs": tag,
    "rules with val Sharpe > 0": f"{(res[f'val_{tag}_sharpe'] > 0).sum()} of {len(res)}",
    f"> buy and hold ({bh_val:+.2f})": f"{(res[f'val_{tag}_sharpe'] > bh_val).sum()} of {len(res)}",
    "median val Sharpe": res[f"val_{tag}_sharpe"].median(),
    "best val Sharpe (chosen on val: optimistic)": res[f"val_{tag}_sharpe"].max(),
    "train-val rank correlation": res[f"train_{tag}_sharpe"].corr(res[f"val_{tag}_sharpe"], method="spearman")}
    for tag in ("net", "gross")])
print(f"best rule per signal, chosen on TRAIN (net of costs, >= 5 train trades; {len(elig)} of {len(res)} rules qualify),"
      " and how it did on validation:")
print(best_per_signal.round(3).to_string(index=False))
print(f"\noverall best on train: {best.signal}, q {best.q}, {best['mode']}, min_hold {best.min_hold} -> "
      f"val Sharpe {best.val_net_sharpe:+.2f}, return {best.val_net_cum_return_pct:+.1f} %, {int(best.val_net_num_trades)} trades, "
      f"costs {best.val_net_costs_pct:.1f} % (buy and hold {bh_val:+.2f}, {BH['val']['cum_return_pct']:+.1f} %)")
print("\nall 252 rules on validation:")
print(dist.round(3).to_string(index=False))
print("\nfor reference: the E16 agents' val Sharpe 1.14 +- 0.27 was CHOSEN on validation (checkpoints), "
      "so it compares with the optimistic 'best val' column, not with the train-chosen rules")

## Figures

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
x = np.arange(len(BINS))
for ax, s in zip(axes, SPLITS):
    d = edge_df[edge_df.split == s]
    for _, r in d.iterrows():
        ax.plot(x, [r[b] for b in BINS], lw=2 if r.signal == "mean" else 1, color="black" if r.signal == "mean" else None,
                marker="o", ms=3, label=f"{r.signal} ({r.horizon_bars} bars)")
    for c in (100 * FR.ROUND_TRIP_COST / 2, -100 * FR.ROUND_TRIP_COST / 2):
        ax.axhline(c, color="grey", ls="--", lw=0.8)
    ax.axhline(0, color="grey", lw=0.5)
    ax.set_xticks(x, BINS, rotation=40, fontsize=8); ax.grid(alpha=0.3)
    ax.set_title(f"{s}: BTC move over the signal's horizon, by signal percentile", fontsize=10)
axes[0].set_ylabel("mean forward return %"); axes[1].legend(fontsize=7, ncol=2)
fig.text(0.5, 0.005, "dashed lines: +-0.12 % = half the round-trip cost; a long-short rule needs the two ends to sit outside them",
         ha="center", fontsize=8)
fig.tight_layout(rect=(0, 0.04, 1, 1)); fig.savefig(REP / "e19_edge.png", dpi=130); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
sub = res[(res["mode"] == "hold") & (res["min_hold"] > 0)]
for ax, tag in zip(axes, ("net", "gross")):
    piv = sub.pivot(index="signal", columns="q", values=f"val_{tag}_sharpe").reindex(list(FR.SIGNALS))
    lim = 3.0                                      # colour scale capped at +-3; the numbers show the real values
    im = ax.imshow(piv.values, cmap="RdBu", vmin=-lim, vmax=lim, aspect="auto")
    ax.set_xticks(range(piv.shape[1]), [f"{q:.2f}" for q in piv.columns]); ax.set_yticks(range(piv.shape[0]), piv.index)
    for (i, k), v in np.ndenumerate(piv.values):
        ax.text(k, i, f"{v:+.1f}", ha="center", va="center", fontsize=7)
    ax.set_xlabel("threshold = train quantile of |signal|")
    ax.set_title(f"validation Sharpe, {'after' if tag == 'net' else 'before'} costs (hold mode, min hold = horizon)", fontsize=10)
    fig.colorbar(im, ax=ax, shrink=0.8)
fig.tight_layout(); fig.savefig(REP / "e19_rules_val_sharpe.png", dpi=130); plt.show()

In [ ]:
g = GRID[int(best.name)]
fig, ax = plt.subplots(figsize=(9, 3.6))
for s_name, pol, st in (("best rule on train", FR.rule_policy([g], NET, "cpu"), "-"), ("buy and hold", buy_and_hold_policy, "--")):
    bt = run_policy(pol, MK["val"], NET)
    ax.plot(R.as_datetime(bt["time_ns"]), bt["equity"][1:], ls=st, lw=1.2, color=None if st == "-" else "black", label=s_name)
ax.set_title(f"validation equity, net of costs: {g['signal']} q {g['q']} {g['mode']} min hold {g['min_hold']}", fontsize=10)
ax.grid(alpha=0.3); ax.legend(fontsize=8); ax.tick_params(axis="x", rotation=30)
fig.tight_layout(); fig.savefig(REP / "e19_best_rule_val_equity.png", dpi=130); plt.show()

In [ ]:
txt = ["# E19: can the market factors be traded after costs, without RL? (train and validation only)\n",
       f"Factors: {F.name} (E15). Round-trip cost {100 * FR.ROUND_TRIP_COST:.2f} %. Buy and hold Sharpe: train "
       f"{BH['train']['sharpe']:+.2f}, val {bh_val:+.2f}. Test and stress_bear not loaded.\n",
       "## 1 · Edge\n", "IC and the gross gain of one round trip on the strongest signals (half spread):\n",
       ic_df.round(4).to_markdown(index=False),
       "\nMean forward return (%) by train percentile of the signal, validation:\n",
       edge_df[edge_df.split == "val"].drop(columns="split").round(3).to_markdown(index=False),
       "\n## 2 · Threshold rules\n", "Best rule per signal, chosen on train (net of costs, at least 5 train trades):\n",
       best_per_signal.round(3).to_markdown(index=False),
       f"\nOverall best on train: {best.signal}, q {best.q}, {best['mode']}, min_hold {best.min_hold}: val Sharpe "
       f"{best.val_net_sharpe:+.2f}, return {best.val_net_cum_return_pct:+.1f} %, {int(best.val_net_num_trades)} trades.\n",
       "All 252 rules on validation:\n", dist.round(3).to_markdown(index=False),
       "\nFigures: e19_edge.png, e19_rules_val_sharpe.png, e19_best_rule_val_equity.png"]
(REP / "e19_factor_rules.md").write_text("\n".join(txt) + "\n", encoding="utf-8")
print("wrote", REP / "e19_factor_rules.md", "|", OUT)

In [ ]:
AUTO_DISCONNECT = True               # release the runtime once the results are on Drive
if AUTO_DISCONNECT and "google.colab" in sys.modules:
    from google.colab import runtime
    runtime.unassign()